In [9]:
# 🔷 Question 1: Build an End-to-End RAG + Agent System (25 Marks)
# 🧩 Scenario
# You are an AI intern at an ed-tech company. Students frequently ask questions about:

# Course policies (refunds, deadlines)
# Lecture content (PDF notes)
# Assignment deadlines
# Their enrollment status
# The company wants a single intelligent assistant that:

# Answers questions using internal documents (PDFs, FAQs)
# Fetches student-specific data (like enrollment or progress) using tools/APIs
# Avoids hallucination and gives reliable answers
# 💻 Task
# Design and implement a working prototype (pseudo-code or real code) for this system.

# Your solution must include:

# ✅ 1. RAG Pipeline
# How you will ingest and preprocess documents
# Chunking strategy (with justification)
# Embedding + vector store choice
# Retrieval logic
# How context is injected into the LLM
# ✅ 2. Agent Integration
# Design an agent that decides:
# When to use RAG
# When to call a tool (e.g., get_student_status(student_id))
# Show how tools are defined and used
# ✅ 3. End-to-End Flow
# Write code or structured pseudo-code showing:

# Input query
# Retrieval step
# Tool calling (if needed)
# Final answer generation
# ✅ 4. Reliability Improvements
# Show at least 2 techniques in code/design to:

# Reduce hallucination
# Improve answer quality
# 🎯 Expected Outcome
# A working architecture/code that demonstrates:

# RAG + Agent working together
# Decision-making capability
# Real-world practicality

In [8]:
# =========================================
# INSTALL LIBRARIES
# =========================================

!pip install faiss-cpu
!pip install groq
!pip install sentence-transformers


# =========================================
# IMPORTS
# =========================================

import os
import faiss
import numpy as np
from groq import Groq
from sentence_transformers import SentenceTransformer
from google.colab import userdata


# =========================================
# API KEY SETUP
# =========================================

GROQ_API_KEY = userdata.get("API_KEY")

print("Loaded Key:", GROQ_API_KEY)

client = Groq(api_key=GROQ_API_KEY)


# =========================================
# DOCUMENTS (IMPORTANT)
# =========================================

documents = [
    "Refund policy: Students can request refund within 7 days of purchase.",
    "Assignments must be submitted before the deadline every Sunday.",
    "Course duration is 3 months.",
    "Certificates are provided after course completion.",
    "Students can track progress in dashboard."
]


# =========================================
# CHUNKING
# =========================================

def chunk_text(text, chunk_size=40):

    words = text.split()

    chunks = [
        " ".join(words[i:i+chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

    return chunks


chunks = []

for doc in documents:
    chunks.extend(chunk_text(doc))


print("Chunks:", chunks)


# =========================================
# EMBEDDINGS + FAISS
# =========================================

embed_model = SentenceTransformer("paraphrase-MiniLM-L3-v2")

embeddings = embed_model.encode(chunks)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))


print("FAISS index ready")


# =========================================
# RETRIEVAL
# =========================================

def retrieve(query, k=2):

    query_embedding = embed_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding), k
    )

    results = []

    for i, d in zip(indices[0], distances[0]):

        if d < 2:
            results.append(chunks[i])

    return results


# =========================================
# GROQ LLM
# =========================================

def generate_answer(query, context):

    if not context:
        return "I don't know based on the course documents."

    prompt = f"""
You are an AI assistant for an ed-tech platform.

Answer ONLY using the provided context.

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(

        model="llama3-8b-8192",

        messages=[
            {"role": "user", "content": prompt}
        ]

    )

    return response.choices[0].message.content


# =========================================
# TOOL (STUDENT DATABASE)
# =========================================

student_db = {

    "101": {"status": "enrolled", "progress": "60%"},
    "102": {"status": "not enrolled", "progress": "0%"},
    "103": {"status": "enrolled", "progress": "85%"}
}


def get_student_status(student_id):

    return student_db.get(student_id, "Student not found")


# =========================================
# AGENT DECISION
# =========================================

def agent_decision(query):

    q = query.lower()

    if "my" in q or "status" in q or "progress" in q:
        return "tool"

    return "rag"


# =========================================
# MAIN SYSTEM
# =========================================

def system(query, student_id=None):

    decision = agent_decision(query)

    print("\n[Agent Decision]:", decision.upper())

    if decision == "tool":

        result = get_student_status(student_id)

        return f"Tool Response → {result}"

    if decision == "rag":

        context = retrieve(query)

        print("[Retrieved Context]:", context)

        answer = generate_answer(query, context)

        return answer


# =========================================
# TEST CASES
# =========================================

print("---- TEST 1 ----")
print(system("What is refund policy?"))

print("\n---- TEST 2 ----")
print(system("When is assignment due?"))

print("\n---- TEST 3 ----")
print(system("What is my enrollment status?", student_id="101"))

print("\n---- TEST 4 ----")
print(system("Tell me course duration"))

Loaded Key: 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
Chunks: ['Refund policy: Students can request refund within 7 days of purchase.', 'Assignments must be submitted before the deadline every Sunday.', 'Course duration is 3 months.', 'Certificates are provided after course completion.', 'Students can track progress in dashboard.']


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index ready
---- TEST 1 ----

[Agent Decision]: RAG
[Retrieved Context]: []
I don't know based on the course documents.

---- TEST 2 ----

[Agent Decision]: RAG
[Retrieved Context]: []
I don't know based on the course documents.

---- TEST 3 ----

[Agent Decision]: TOOL
Tool Response → {'status': 'enrolled', 'progress': '60%'}

---- TEST 4 ----

[Agent Decision]: RAG
[Retrieved Context]: []
I don't know based on the course documents.
